In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from featurewiz import featurewiz

current_dir = Path.cwd()
project_root = current_dir.parents[2]
path_data = project_root / 'DATA_PPMI/Results/MODEL_DATA/data_bl_v08.csv'
data = pd.read_csv(path_data)

def HY_classification(nhy):
    if nhy == 0:
        return 'Healthy'
    elif nhy in [1, 2]:
        return 'Early Stage'
    else:
        return 'Advanced Stage'

data['STAGE'] = data['NHY'].apply(HY_classification)
cols_drop=['ENRLLRRK2','ENRLGBA','COHORT_DEFINITION','SEX','RAWHITE','EDUCYRS','AGE_AT_VISIT']

data_stats = (
    data.drop(columns=cols_drop)
        .groupby(level='PATNO')
        .agg(['mean','min','max','var','std'])
)

data_stats.columns = ['_'.join(col) for col in data_stats.columns]


# --- codifica el target a enteros ---
le = LabelEncoder()
data['STAGE_ENC'] = le.fit_transform(data['STAGE'])  # 0,1,2

# (opcional) quita STAGE string para que no “confunda”
# data = data.drop(columns=['STAGE'])

target = "STAGE_ENC"

selected_features, df_selected = featurewiz(
    data,
    target=target,
    corr_limit=0.90,
    verbose=1
)

print("Clases:", list(le.classes_))  # para saber qué entero es cada clase
print("Features seleccionadas:", selected_features)

X_final = df_selected.drop(columns=[target])
y_final = df_selected[target]

############################################################################################
############       F A S T   F E A T U R E  E N G G    A N D    S E L E C T I O N ! ########
# Be judicious with featurewiz. Don't use it to create too many un-interpretable features! #
############################################################################################
featurewiz has selected 0.9 as the correlation limit. Change this limit to fit your needs...
    Skipping feature engineering since no feature_engg input...
Skipping category encoding since no category encoders specified in input...
    Single_Label Multi_Classification problem 
    Loaded train data. Shape = (2012, 197)
    Single_Label Multi_Classification problem 
No test data filename given...
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
###################################################